In [ ]:
#诈骗分类框架人工编码一致性分析脚本

import pandas as pd
import numpy as np
import ast
from sklearn.metrics import cohen_kappa_score

# ==================== 1. 数据加载 ====================
def load_data_robust(file_path):
    """读取Excel，通过位置提取标签列和sequence列"""
    # 读取时不处理表头，保留原始数据
    df_raw = pd.read_excel(file_path, header=None)
    
    # 前两行是表头，第三行开始是数据
    header_row1 = df_raw.iloc[0].fillna('').astype(str)
    header_row2 = df_raw.iloc[1].fillna('').astype(str)
    
    # 构建列名：结合两层表头（避免Unnamed）
    columns = []
    for c1, c2 in zip(header_row1, header_row2):
        if c2 and 'Unnamed' not in c2:
            columns.append(c2)
        elif c1 and 'Unnamed' not in c1:
            columns.append(c1)
        else:
            columns.append(f"Col_{len(columns)}")
    
    # 数据部分（从第3行开始）
    data = df_raw.iloc[2:].copy()
    data.columns = columns
    data.reset_index(drop=True, inplace=True)
    
    print(f"\n文件 {file_path} 的实际列名（前30字符）:")
    for i, col in enumerate(columns):
        print(f"  {i}: {col[:30]}")
    
    # 定位关键列
    # Sample ID 通常在 A 列（索引0）
    sample_id_col = columns[0] if 'Sample' in columns[0] else columns[0]
    
    # 标签列：根据Codebook，标签列从 D 列开始，连续16列（G1, G2, G3, L1, L2, L3, R1, R2, R3, C1, C2, C3, C4, C5, OT, NS）
    # 在Excel中，对应列索引为 3 到 18（A=0, B=1, C=2, D=3...）
    label_start_idx = 3   # D列
    label_end_idx = 18    # S列（共16列）
    
    # 提取标签数据
    label_data = data.iloc[:, label_start_idx:label_end_idx+1]
    # 将可能的空值或非数值转为0
    label_data = label_data.apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
    label_data.columns = ['G1','G2','G3','L1','L2','L3','R1','R2','R3',
                          'C1','C2','C3','C4','C5','OT','NS']
    
    # Sample ID
    sample_ids = data.iloc[:, 0]
    
    # Sequence 列：索引为 19（T列）
    sequence_raw = data.iloc[:, 19] if data.shape[1] > 19 else pd.Series([''] * len(data))
    
    # 解析sequence
    def parse_seq(s):
        if pd.isna(s) or str(s).strip() in ('', '[]', 'nan'):
            return None
        try:
            seq_list = ast.literal_eval(str(s))
            return seq_list if isinstance(seq_list, list) else None
        except:
            return None
    
    sequence_parsed = sequence_raw.apply(parse_seq)
    
    return label_data, sample_ids, sequence_parsed, sequence_raw

print("正在加载编码结果...")
label_a, ids_a, seq_a, seq_raw_a = load_data_robust('人工编码3-A.xlsx')
label_b, ids_b, seq_b, seq_raw_b = load_data_robust('人工编码3-B.xlsx')

# 确保样本数一致
assert len(label_a) == len(label_b), "两个文件的样本数不一致"
n = len(label_a)
print(f"\n样本总数: {n}")

# ==================== 2. 二值标签一致性分析 ====================
full_match_labels = (label_a.values == label_b.values).all(axis=1)
label_accuracy = full_match_labels.mean()
print(f"\n【二值标签】整体完全一致率: {label_accuracy:.2%} ({full_match_labels.sum()}/{n})")

label_cols = label_a.columns.tolist()
kappa_results = []
for col in label_cols:
    y_a = label_a[col].values
    y_b = label_b[col].values
    kappa = cohen_kappa_score(y_a, y_b)
    kappa_results.append({
        '标签': col,
        'Kappa': round(kappa, 4),
        'A=1比例': y_a.mean(),
        'B=1比例': y_b.mean(),
        '一致样本数': (y_a == y_b).sum()
    })
kappa_df = pd.DataFrame(kappa_results).sort_values('Kappa')
print("\n逐标签Cohen's Kappa (升序):")
print(kappa_df.to_string(index=False))

# ==================== 3. Sequence字段一致性分析 ====================
def seq_to_str(seq):
    if seq is None:
        return "[]"
    return str(seq)

seq_str_a = [seq_to_str(s) for s in seq_a]
seq_str_b = [seq_to_str(s) for s in seq_b]
seq_match = [a == b for a, b in zip(seq_str_a, seq_str_b)]
seq_accuracy = np.mean(seq_match)
print(f"\n【sequence字段】完全匹配率: {seq_accuracy:.2%} ({sum(seq_match)}/{n})")

seq_nonempty_a = sum(1 for s in seq_a if s is not None)
seq_nonempty_b = sum(1 for s in seq_b if s is not None)
print(f"A编码非空sequence数: {seq_nonempty_a} ({seq_nonempty_a/n:.1%})")
print(f"B编码非空sequence数: {seq_nonempty_b} ({seq_nonempty_b/n:.1%})")

# ==================== 4. 综合不一致样本导出 ====================
diff_records = []
for i in range(n):
    sample_id = ids_a.iloc[i]
    label_diff_cols = []
    for col in label_cols:
        if label_a.iloc[i][col] != label_b.iloc[i][col]:
            label_diff_cols.append(f"{col}(A:{label_a.iloc[i][col]},B:{label_b.iloc[i][col]})")
    seq_diff = "" if seq_match[i] else f"seq(A:{seq_raw_a.iloc[i]},B:{seq_raw_b.iloc[i]})"
    if label_diff_cols or not seq_match[i]:
        diff_records.append({
            '行号': i+1,
            'Sample ID': sample_id,
            '标签差异': '; '.join(label_diff_cols) if label_diff_cols else '无',
            'Sequence差异': seq_diff,
            '标签差异数': len(label_diff_cols)
        })

diff_df = pd.DataFrame(diff_records)
diff_df.to_excel('不一致样本清单_含sequence.xlsx', index=False)
print(f"\n综合不一致样本数: {len(diff_df)}，已保存至 不一致样本清单_含sequence.xlsx")

# ==================== 5. 生成报告 ====================
low_kappa = kappa_df[kappa_df['Kappa'] < 0.6]['标签'].tolist()
report = f"""
=================== 编码信度分析报告 ===================

样本总数: {n}

【1. 二值标签一致性】
  整体完全一致率: {label_accuracy:.2%} ({full_match_labels.sum()}/{n})
  存在标签分歧样本数: {sum(~full_match_labels)}

  逐标签Cohen's Kappa:
{kappa_df.to_string(index=False)}

  低信度标签 (Kappa < 0.6): {', '.join(low_kappa) if low_kappa else '无'}

【2. Sequence字段一致性】
  序列完全匹配率: {seq_accuracy:.2%} ({sum(seq_match)}/{n})
  A编码非空序列: {seq_nonempty_a} 条
  B编码非空序列: {seq_nonempty_b} 条

【3. 综合分歧样本】
  标签或序列任一不一致的样本总数: {len(diff_df)}
  其中仅标签不一致: {sum((diff_df['标签差异'] != '无') & (diff_df['Sequence差异'] == ''))}
  仅序列不一致: {sum((diff_df['标签差异'] == '无') & (diff_df['Sequence差异'] != ''))}
  两者都不一致: {sum((diff_df['标签差异'] != '无') & (diff_df['Sequence差异'] != ''))}

【4. 效度分析建议】
  - 低Kappa标签需重点复核Codebook定义。
  - 检查Sequence不一致样本，优化时序判断规则。
  - 详细分歧见: 不一致样本清单_含sequence.xlsx
"""
print(report)
with open('信度分析报告_含sequence.txt', 'w', encoding='utf-8') as f:
    f.write(report)

正在加载编码结果...

文件 人工编码3-A.xlsx 的实际列名（前30字符）:
  0: Sample ID
  1: title
  2: content
  3: 财富诱饵
  4: 关系诱饵
  5: 机会诱饵
  6: 安全威胁
  7: 财产威胁
  8: 隐私威胁
  9: 冒充亲友
  10: 冒充权威
  11: 冒充客服/平台
  12: 误点击
  13: 误输入
  14: 误识别
  15: 误触发
  16: 远程控制
  17: 其他诈骗
  18: 信息不足
  19: sequence
  20: 说明

文件 人工编码3-B.xlsx 的实际列名（前30字符）:
  0: Sample ID
  1: title
  2: content
  3: 财富诱饵
  4: 关系诱饵
  5: 机会诱饵
  6: 安全威胁
  7: 财产威胁
  8: 隐私威胁
  9: 冒充亲友
  10: 冒充权威
  11: 冒充客服/平台
  12: 误点击
  13: 误输入
  14: 误识别
  15: 误触发
  16: 远程控制
  17: 其他诈骗
  18: 信息不足
  19: sequence
  20: 说明

样本总数: 500

【二值标签】整体完全一致率: 94.60% (473/500)

逐标签Cohen's Kappa (升序):
标签  Kappa  A=1比例  B=1比例  一致样本数
OT 0.8205  0.018  0.016    497
L2 0.8562  0.008  0.006    499
R2 0.8626  0.032  0.028    496
L1 0.8879  0.010  0.008    499
NS 0.9144  0.624  0.632    480
G1 0.9571  0.108  0.100    496
C1 0.9667  0.032  0.030    499
C2 0.9667  0.032  0.030    499
R3 0.9814  0.058  0.056    499
G2 1.0000  0.014  0.014    500
G3 1.0000  0.022  0.022    500
L3 1.0000  0.006  0.006 

In [1]:
#提取llm标注结果中匹配人工标注样本的列表与结果
import pandas as pd

# ---------- 1. 提取 Sample ID 列表 ----------
df_raw = pd.read_excel('人工编码3-B.xlsx', sheet_name='Sheet1', header=None)
sample_ids = df_raw.iloc[2:, 0].dropna().astype(str).str.strip().tolist()
print(f"Total sample_ids: {len(sample_ids)}")

# ---------- 2. 读取两个待匹配文件 ----------
df1 = pd.read_excel('llm_posts_output_classified.xlsx', sheet_name='output_classified')
df2 = pd.read_excel('llm_news_output_classified.xlsx', sheet_name='news_output1_classified')

# 查看完整列名，便于核对
print("df1 columns:", df1.columns.tolist())
print("df2 columns:", df2.columns.tolist())

# ---------- 3. 清洗 ID 列 ----------
df1['post_id'] = df1['post_id'].astype(str).str.strip()
df2['news_id'] = df2['news_id'].astype(str).str.strip()

# ---------- 4. 统一列名：将 df2 的 ID 列改为 post_id ----------
df2 = df2.rename(columns={'news_id': 'post_id'})

# ---------- 5. 筛选匹配行，并对每个 ID 去重（保留第一条） ----------
df1_matched = df1[df1['post_id'].isin(sample_ids)].drop_duplicates(subset='post_id', keep='first').copy()
df2_matched = df2[df2['post_id'].isin(sample_ids)].drop_duplicates(subset='post_id', keep='first').copy()

print(f"df1 unique matched rows: {len(df1_matched)}")
print(f"df2 unique matched rows: {len(df2_matched)}")

# ---------- 6. 确定要保留的列（除 post_id 外的所有列）----------
# 找出两个文件共有的列（避免合并时出现重复列）
common_cols = list(set(df1_matched.columns) & set(df2_matched.columns))
# 移除 post_id（因为它将作为连接键）
common_cols.remove('post_id')

# ---------- 7. 合并：以 sample_ids 为基准，左连接 df1 和 df2 ----------
base = pd.DataFrame({'post_id': sample_ids})

# 先合并 df1（带后缀 _df1）
merged = base.merge(df1_matched, on='post_id', how='left', suffixes=('', '_df1'))
# 再合并 df2（带后缀 _df2），对于重复列会自动加后缀
merged = merged.merge(df2_matched, on='post_id', how='left', suffixes=('', '_df2'))

# ---------- 8. 对于共有列，优先使用 df1 的值，缺失则用 df2 补全 ----------
for col in common_cols:
    col_df1 = col + '_df1' if col + '_df1' in merged.columns else col
    col_df2 = col + '_df2' if col + '_df2' in merged.columns else col
    if col_df1 in merged.columns and col_df2 in merged.columns:
        merged[col] = merged[col_df1].combine_first(merged[col_df2])
    elif col_df1 in merged.columns:
        merged[col] = merged[col_df1]
    elif col_df2 in merged.columns:
        merged[col] = merged[col_df2]

# 添加来源标记
merged['source'] = 'posts'
merged.loc[merged['post_id'].isin(df2_matched['post_id']) & ~merged['post_id'].isin(df1_matched['post_id']), 'source'] = 'news'
merged.loc[merged['post_id'].isin(df1_matched['post_id']) & merged['post_id'].isin(df2_matched['post_id']), 'source'] = 'both'

# ---------- 9. 删除临时后缀列 ----------
suffix_cols = [c for c in merged.columns if c.endswith('_df1') or c.endswith('_df2')]
merged.drop(columns=suffix_cols, inplace=True)

# ---------- 10. 保存结果 ----------
merged.to_excel('merged_full_columns.xlsx', index=False)
print(f"Final rows: {len(merged)}")
print("Output columns:", merged.columns.tolist())

Total sample_ids: 500
df1 columns: ['样本ID', 'G1', 'G2', 'G3', 'L1', 'L2', 'L3', 'R1', 'R2', 'R3', 'C1', 'C2', 'C3', 'C4', 'C5', 'OT', 'NS', '判断依据', '一级分类', '二级分类', '三级分类', '界面标签', '最终标签', 'title', 'content', 'alltext', 'time_series', 'post_id']
df2 columns: ['样本ID', 'G1', 'G2', 'G3', 'L1', 'L2', 'L3', 'R1', 'R2', 'R3', 'C1', 'C2', 'C3', 'C4', 'C5', 'OT', 'NS', '判断依据', '一级分类', '二级分类', '三级分类', '界面标签', '最终标签', 'title', 'content', 'alltext', 'time_series', 'news_id']
df1 unique matched rows: 325
df2 unique matched rows: 175
Final rows: 500
Output columns: ['post_id', '样本ID', 'G1', 'G2', 'G3', 'L1', 'L2', 'L3', 'R1', 'R2', 'R3', 'C1', 'C2', 'C3', 'C4', 'C5', 'OT', 'NS', '判断依据', '一级分类', '二级分类', '三级分类', '界面标签', '最终标签', 'title', 'content', 'alltext', 'time_series', 'source']


In [2]:
#计算人工标注与llm标注的一致性、准确率等，检验标签结果质量
import sys
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, precision_recall_fscore_support, accuracy_score
import ast
import warnings
warnings.filterwarnings('ignore')

# ==================== 配置 ====================
FILE_A = "人工编码3-A.xlsx"
FILE_B = "人工编码3-B.xlsx"
FILE_LLM = "llm_merged_columns.xlsx"

LABEL_COLS = ['G1', 'G2', 'G3', 'L1', 'L2', 'L3', 'R1', 'R2', 'R3',
              'C1', 'C2', 'C3', 'C4', 'C5', 'OT', 'NS', 'sequence']
BINARY_COLS = [c for c in LABEL_COLS if c != 'sequence']

# ==================== 工具函数 ====================
def parse_label(val):
    """将单元格值解析为规范化集合"""
    if pd.isna(val):
        return set()
    # 处理数值类型 0/1
    if isinstance(val, (int, float)):
        if val == 1:
            return {'1'}
        else:
            return set()
    s = str(val).strip()
    if s in ('0', 'nan', ''):
        return set()
    # 处理列表格式如 ["R3"] 或 ['G1','G3']
    if s.startswith('[') and s.endswith(']'):
        try:
            lst = ast.literal_eval(s)
            if isinstance(lst, list):
                return set(str(x).strip() for x in lst if str(x).strip())
        except:
            pass
    # 逗号分隔格式
    if ',' in s:
        return set(x.strip() for x in s.split(',') if x.strip())
    if s == '1':
        return {'1'}
    if s in LABEL_COLS:
        return {s}
    return {s}

def is_positive(label_set):
    return len(label_set) > 0

def load_manual_table(filepath):
    """加载人工编码表，动态适配列数"""
    df = pd.read_excel(filepath, header=None, skiprows=2)
    n_cols = df.shape[1]
    print(f"文件 {filepath} 实际列数：{n_cols}")
    base_cols = ['Sample ID', 'title', 'content'] + LABEL_COLS  # 3+17=20
    remaining = n_cols - len(base_cols)
    if remaining == 1:
        cols = base_cols + ['note']
    elif remaining == 2:
        cols = base_cols + ['sequence_orig', 'note']
    else:
        cols = base_cols + [f'extra_{i}' for i in range(remaining)]
    df.columns = cols
    df['Sample ID'] = df['Sample ID'].astype(str).str.strip()
    return df

def load_llm_table(filepath):
    """加载LLM结果表，处理列名差异，使用'post_id'作为匹配ID"""
    df = pd.read_excel(filepath, header=0)
    print(f"LLM表原始列名：{list(df.columns)}")
    
    if 'post_id' not in df.columns:
        raise KeyError("LLM表中未找到'post_id'列，无法与人工表匹配。")
    
    df = df.rename(columns={'post_id': 'Sample ID'})
    df['Sample ID'] = df['Sample ID'].astype(str).str.strip()
    
    # 列名映射：将LLM中的time_series映射为sequence
    column_mapping = {'time_series': 'sequence'}
    df = df.rename(columns=column_mapping)
    return df

def compare_labels(row_a, row_b, cols):
    for col in cols:
        if parse_label(row_a[col]) != parse_label(row_b[col]):
            return False
    return True

# ==================== 步骤1：黄金标准 ====================
print("=" * 50)
print("步骤1：提取编码员完全一致的样本作为黄金标准")
print("=" * 50)

df_a = load_manual_table(FILE_A)
df_b = load_manual_table(FILE_B)

common_ids = set(df_a['Sample ID']).intersection(set(df_b['Sample ID']))
print(f"编码员A样本数：{len(df_a)}，编码员B样本数：{len(df_b)}")
print(f"两表共有的Sample ID数量：{len(common_ids)}")

if len(common_ids) == 0:
    print("错误：两份人工编码表无交集，请检查ID格式。")
    sys.exit(1)

df_a = df_a[df_a['Sample ID'].isin(common_ids)].sort_values('Sample ID').reset_index(drop=True)
df_b = df_b[df_b['Sample ID'].isin(common_ids)].sort_values('Sample ID').reset_index(drop=True)

consistent_mask = []
for i in range(len(df_a)):
    consistent_mask.append(compare_labels(df_a.iloc[i], df_b.iloc[i], LABEL_COLS))

df_gold = df_a[consistent_mask].copy()
print(f"完全一致的样本数：{len(df_gold)}（占交集总数的 {len(df_gold)/len(df_a)*100:.1f}%）")

if len(df_gold) == 0:
    print("错误：未找到任何完全一致的样本，无法生成黄金标准。")
    sys.exit(1)

df_gold.to_excel("gold_standard.xlsx", index=False)
print("已保存黄金标准文件：gold_standard.xlsx")

# ==================== 步骤2：LLM对齐与评估 ====================
print("\n" + "=" * 50)
print("步骤2：LLM结果与黄金标准对齐，计算逐标签Kappa及序列匹配率")
print("=" * 50)

df_llm = load_llm_table(FILE_LLM)
print(f"LLM表总记录数：{len(df_llm)}")

# 检查LLM标签列内容（调试用）
print("\nLLM表标签列样例值（前3行）：")
for col in LABEL_COLS:
    if col in df_llm.columns:
        sample_vals = df_llm[col].head(3).tolist()
        print(f"  {col}: {sample_vals}")
    else:
        print(f"  警告：LLM表中缺少列 '{col}'")

common_ids_llm = set(df_gold['Sample ID']).intersection(set(df_llm['Sample ID']))
print(f"\n黄金标准与LLM表共有的Sample ID数量：{len(common_ids_llm)}")

if len(common_ids_llm) == 0:
    print("错误：黄金标准与LLM结果无交集，无法进行评估。")
    print("黄金标准前5个ID:", df_gold['Sample ID'].head().tolist())
    print("LLM表前5个ID:", df_llm['Sample ID'].head().tolist())
    sys.exit(1)

llm_aligned = df_llm[df_llm['Sample ID'].isin(common_ids_llm)].copy()
llm_aligned = llm_aligned.sort_values('Sample ID').reset_index(drop=True)
gold_aligned = df_gold[df_gold['Sample ID'].isin(common_ids_llm)].copy()
gold_aligned = gold_aligned.sort_values('Sample ID').reset_index(drop=True)

print(f"对齐后的共同样本数：{len(gold_aligned)}")
if len(gold_aligned) == 0:
    print("错误：对齐后样本数为0，终止评估。")
    sys.exit(1)

llm_aligned.to_excel("llm_aligned_to_gold.xlsx", index=False)
print("已保存LLM对齐结果：llm_aligned_to_gold.xlsx")

# 初始化结果列表
results = []
print("\n逐标签评估结果（基于二值化）：")
print("-" * 80)
print(f"{'标签':<6} {'Kappa':<8} {'准确率':<8} {'精确率':<8} {'召回率':<8} {'F1':<8} {'支持度':<8}")
print("-" * 80)

for col in BINARY_COLS:
    if col not in gold_aligned.columns or col not in llm_aligned.columns:
        print(f"警告：列 '{col}' 在黄金标准或LLM表中不存在，跳过。")
        continue
    
    y_true = gold_aligned[col].apply(lambda x: 1 if is_positive(parse_label(x)) else 0)
    y_pred = llm_aligned[col].apply(lambda x: 1 if is_positive(parse_label(x)) else 0)
    
    if len(y_true) == 0 or len(y_pred) == 0:
        print(f"警告：标签 '{col}' 数据为空，跳过。")
        continue
    
    try:
        kappa = cohen_kappa_score(y_true, y_pred)
    except ValueError:
        kappa = np.nan
    
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )
    pos_support = y_true.sum()
    
    results.append({
        '标签': col,
        'Kappa': kappa,
        '准确率': acc,
        '精确率': precision,
        '召回率': recall,
        'F1': f1,
        '人工正样本数': pos_support,
        'LLM正样本数': y_pred.sum()
    })
    print(f"{col:<6} {kappa if not np.isnan(kappa) else 'N/A':<8} {acc:<8.3f} {precision:<8.3f} {recall:<8.3f} {f1:<8.3f} {pos_support:<8}")

# 处理sequence列
if 'sequence' in gold_aligned.columns and 'sequence' in llm_aligned.columns:
    seq_gold = gold_aligned['sequence'].apply(parse_label)
    seq_llm = llm_aligned['sequence'].apply(parse_label)
    exact_match = (seq_gold == seq_llm)
    seq_acc = exact_match.mean()
    jaccard_sim = np.mean([
        len(g & p) / len(g | p) if len(g | p) > 0 else 1.0
        for g, p in zip(seq_gold, seq_llm)
    ])
    print("-" * 80)
    print(f"sequence列完全匹配率: {seq_acc:.3f}")
    print(f"sequence列平均Jaccard相似度: {jaccard_sim:.3f}")
else:
    print("警告：未找到sequence列，跳过序列评估。")

if results:
    df_results = pd.DataFrame(results)
    df_results.to_excel("label_kappa_report.xlsx", index=False)
    print("\n已保存：label_kappa_report.xlsx")
else:
    print("\n警告：未生成任何标签评估结果。")

# ==================== 步骤3：偏差报告 ====================
print("\n" + "=" * 50)
print("步骤3：生成偏差报告")
print("=" * 50)

if len(gold_aligned) == 0:
    print("无有效样本，跳过偏差报告生成。")
    sys.exit(0)

bias_report = []
for col in BINARY_COLS:
    if col not in gold_aligned.columns or col not in llm_aligned.columns:
        continue
    y_true = gold_aligned[col].apply(lambda x: 1 if is_positive(parse_label(x)) else 0)
    y_pred = llm_aligned[col].apply(lambda x: 1 if is_positive(parse_label(x)) else 0)
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    total_pos_true = y_true.sum()
    over_label_rate = fp / total_pos_true if total_pos_true > 0 else 0
    under_label_rate = fn / total_pos_true if total_pos_true > 0 else 0
    bias_report.append({
        '标签': col,
        '过标数(FP)': fp,
        '漏标数(FN)': fn,
        '正确标正(TP)': tp,
        '正确标负(TN)': tn,
        '过标率(FP/人工正)': over_label_rate,
        '漏标率(FN/人工正)': under_label_rate,
        '偏差倾向': '过标倾向' if fp > fn else ('漏标倾向' if fn > fp else '平衡')
    })

if bias_report:
    df_bias = pd.DataFrame(bias_report)
    print("\n偏差报告概览：")
    print(df_bias[['标签', '过标数(FP)', '漏标数(FN)', '过标率(FP/人工正)', '漏标率(FN/人工正)', '偏差倾向']].to_string(index=False))
    df_bias.to_excel("bias_report.xlsx", index=False)
    print("\n已保存：bias_report.xlsx")

    over_labels = df_bias[df_bias['偏差倾向'] == '过标倾向']['标签'].tolist()
    under_labels = df_bias[df_bias['偏差倾向'] == '漏标倾向']['标签'].tolist()
    if over_labels:
        print(f"LLM倾向于过标的标签：{over_labels}")
    if under_labels:
        print(f"LLM倾向于漏标的标签：{under_labels}")
else:
    print("未生成偏差报告（无有效标签数据）。")

print("\n所有步骤完成。")

步骤1：提取编码员完全一致的样本作为黄金标准
文件 人工编码3-A.xlsx 实际列数：21
文件 人工编码3-B.xlsx 实际列数：21
编码员A样本数：500，编码员B样本数：500
两表共有的Sample ID数量：500
完全一致的样本数：463（占交集总数的 92.6%）
已保存黄金标准文件：gold_standard.xlsx

步骤2：LLM结果与黄金标准对齐，计算逐标签Kappa及序列匹配率
LLM表原始列名：['post_id', '样本ID', 'G1', 'G2', 'G3', 'L1', 'L2', 'L3', 'R1', 'R2', 'R3', 'C1', 'C2', 'C3', 'C4', 'C5', 'OT', 'NS', '判断依据', '一级分类', '二级分类', '三级分类', '界面标签', '最终标签', 'title', 'content', 'alltext', 'time_series', 'source']
LLM表总记录数：500

LLM表标签列样例值（前3行）：
  G1: [0, 0, 0]
  G2: [0, 1, 0]
  G3: [0, 0, 0]
  L1: [0, 1, 0]
  L2: [0, 0, 1]
  L3: [0, 0, 0]
  R1: [0, 0, 0]
  R2: [0, 0, 0]
  R3: [0, 0, 0]
  C1: [0, 0, 0]
  C2: [0, 0, 0]
  C3: [0, 0, 0]
  C4: [0, 0, 0]
  C5: [0, 0, 0]
  OT: [0, 0, 0]
  NS: [0, 0, 0]
  sequence: [nan, '["G2", "L1"]', nan]

黄金标准与LLM表共有的Sample ID数量：463
对齐后的共同样本数：463
已保存LLM对齐结果：llm_aligned_to_gold.xlsx

逐标签评估结果（基于二值化）：
--------------------------------------------------------------------------------
标签     Kappa    准确率      精确率      召回率      F1       支持度     
